# Análise Exploratória de Dados — Passos Mágicos (PEDE)

Este notebook apresenta uma análise exploratória completa dos dados do programa **Passos Mágicos**, contidos na planilha `data/raw.xlsx`. O objetivo principal é compreender o perfil dos alunos, analisar indicadores de desempenho e identificar padrões relacionados à **defasagem escolar**.

### Estrutura do notebook
1. **Carregamento e visão geral** — importação, dimensões, tipos e amostras
2. **Qualidade dos dados** — valores ausentes e inconsistências
3. **Perfil demográfico** — gênero, idade, escola, tempo no programa
4. **Indicadores de desempenho** — INDE, IAA, IEG, IPS, IDA, IPV, IAN
5. **Disciplinas** — notas de Matemática, Português e Inglês
6. **Análise da variável-alvo** — distribuição de `Defas` / nível de defasagem
7. **Correlações e relações multivariadas**
8. **Evolução entre anos (Pedras)** — progressão dos alunos
9. **Comparação entre abas (2022, 2023, 2024)**
10. **Conclusões e próximos passos**

---
## 1. Carregamento e Visão Geral dos Dados

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações visuais
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Bibliotecas carregadas com sucesso.')

In [ ]:
# Carregar todas as abas da planilha
sheets = pd.read_excel('data/raw.xlsx', sheet_name=None)
print(f'Abas encontradas: {list(sheets.keys())}')
for name, df_sheet in sheets.items():
    print(f'  {name}: {df_sheet.shape[0]} linhas × {df_sheet.shape[1]} colunas')

Vamos focar nossa análise exploratória na aba **PEDE2022**, que é a base utilizada para treinar o modelo de predição de defasagem escolar. As abas de 2023 e 2024 possuem colunas ligeiramente diferentes e serão analisadas comparativamente ao final.

In [ ]:
# Trabalhar com a aba principal
df = sheets['PEDE2022'].copy()
print(f'Shape: {df.shape}')
print(f'Colunas ({len(df.columns)}): {list(df.columns)}')

In [ ]:
# Primeiras linhas
df.head()

In [ ]:
# Tipos de dados
df.dtypes

In [ ]:
# Resumo estatístico das variáveis numéricas
df.describe().T

In [ ]:
# Resumo das variáveis categóricas
df.describe(include='object').T

---
## 2. Qualidade dos Dados — Valores Ausentes

In [ ]:
# Contagem e percentual de valores ausentes por coluna
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Ausentes': missing, '% Ausentes': missing_pct})
missing_df = missing_df[missing_df['Ausentes'] > 0].sort_values('Ausentes', ascending=False)
print(f'Colunas com dados ausentes: {len(missing_df)} de {len(df.columns)}')
missing_df

In [ ]:
# Visualização do mapa de dados ausentes
fig, ax = plt.subplots(figsize=(14, 6))
cols_with_missing = missing_df.index.tolist()
sns.heatmap(df[cols_with_missing].isnull().T, cbar=False, cmap='YlOrRd',
            yticklabels=True, ax=ax)
ax.set_title('Mapa de Valores Ausentes (colunas com missing)', fontsize=14)
ax.set_xlabel('Observações')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de barras dos valores ausentes
fig, ax = plt.subplots(figsize=(10, 5))
missing_df['% Ausentes'].plot(kind='barh', color=sns.color_palette('YlOrRd_r', len(missing_df)), ax=ax)
ax.set_title('Percentual de Valores Ausentes por Coluna')
ax.set_xlabel('% Ausente')
for i, v in enumerate(missing_df['% Ausentes']):
    ax.text(v + 0.5, i, f'{v}%', va='center', fontsize=10)
plt.tight_layout()
plt.show()

**Observações:**
- **Inglês** possui ~67% de dados ausentes — a disciplina provavelmente não é oferecida para todas as fases.
- **Pedra 20** (~62%) e **Pedra 21** (~46%) têm muitos ausentes pois nem todos os alunos estavam no programa nos anos anteriores.
- **Avaliador4** e **Rec Av4** (~64%) indicam que a maioria dos alunos passou por no máximo 3 avaliadores.
- **Avaliador3** (~38%) confirma que muitos alunos tiveram apenas 2 avaliadores.
- **Matem** e **Portug** possuem apenas 2 registros ausentes — podem ser tratados facilmente com imputação.

---
## 3. Perfil Demográfico dos Alunos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 3.1 Distribuição por Gênero
colors_genero = ['#e76f51', '#2a9d8f']
df['Gênero'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0],
                                  colors=colors_genero, startangle=90)
axes[0].set_title('Distribuição por Gênero')
axes[0].set_ylabel('')

# 3.2 Distribuição por Instituição de Ensino
df['Instituição de ensino'].value_counts().plot(kind='bar', ax=axes[1],
                                                  color=sns.color_palette('Set2', 3))
axes[1].set_title('Instituição de Ensino')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# 3.3 Distribuição por Fase
df['Fase'].value_counts().sort_index().plot(kind='bar', ax=axes[2],
                                            color=sns.color_palette('Blues_d', df['Fase'].nunique()))
axes[2].set_title('Distribuição por Fase')
axes[2].set_xlabel('Fase')
axes[2].tick_params(axis='x', rotation=0)
for p in axes[2].patches:
    axes[2].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=10)

plt.suptitle('Perfil Demográfico dos Alunos', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3.4 Distribuição de Idade
sns.histplot(df['Idade 22'], bins=15, kde=True, ax=axes[0], color='#2a9d8f')
axes[0].set_title('Distribuição de Idade (2022)')
axes[0].set_xlabel('Idade')
axes[0].axvline(df['Idade 22'].median(), color='red', linestyle='--', label=f'Mediana: {df["Idade 22"].median():.0f}')
axes[0].legend()

# 3.5 Ano de ingresso no programa
df['Ano ingresso'].value_counts().sort_index().plot(kind='bar', ax=axes[1],
                                                     color=sns.color_palette('viridis', df['Ano ingresso'].nunique()))
axes[1].set_title('Ano de Ingresso no Programa')
axes[1].set_xlabel('Ano')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 3.6 Tempo no programa (feature derivada)
df['tempo_no_programa'] = 2022 - df['Ano ingresso']

fig, ax = plt.subplots(figsize=(10, 4))
sns.countplot(x='tempo_no_programa', data=df, palette='viridis', ax=ax)
ax.set_title('Tempo no Programa (anos até 2022)')
ax.set_xlabel('Anos no programa')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

**Observações:**
- A base é composta por **53% meninas** e **47% meninos** — distribuição relativamente equilibrada.
- A grande maioria dos alunos estuda em **Escola Pública** (~87%).
- As **Fases 0–3** concentram a maior parte dos alunos (fases iniciais do programa).
- A mediana de idade é **12 anos**, com distribuição entre 7 e 21 anos.
- Muitos alunos ingressaram em **2021/2022**, indicando uma base com bastante renovação recente.

---
## 4. Indicadores de Desempenho

In [ ]:
# Indicadores numéricos de interesse
indicadores = ['INDE 22', 'IAA', 'IEG', 'IPS', 'IDA', 'IPV', 'IAN']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(indicadores):
    sns.histplot(df[col].dropna(), bins=25, kde=True, ax=axes[i], color=sns.color_palette('tab10')[i])
    axes[i].set_title(col)
    axes[i].axvline(df[col].median(), color='red', linestyle='--', alpha=0.7)

# Remover o subplot extra
axes[-1].set_visible(False)

plt.suptitle('Distribuição dos Indicadores de Desempenho', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots dos indicadores
fig, ax = plt.subplots(figsize=(12, 5))
df[indicadores].plot(kind='box', ax=ax, patch_artist=True,
                     boxprops=dict(facecolor='#a8dadc', alpha=0.7))
ax.set_title('Boxplots dos Indicadores de Desempenho')
ax.set_ylabel('Valor')
plt.tight_layout()
plt.show()

In [ ]:
# Estatísticas descritivas dos indicadores
df[indicadores].describe().T.round(2)

**Observações:**
- O **INDE 22** (índice geral de desempenho) tem média ≈ 7.04 e dispersão moderada (std ≈ 1.02).
- **IAA** (Indicador de Auto-Avaliação) tende a ser alto (média ≈ 8.3), indicando que os alunos se autoavaliam positivamente.
- **IDA** (Indicador de Desempenho Acadêmico) tem a maior dispersão (std ≈ 2.05), revelando heterogeneidade no desempenho.
- **IAN** apresenta distribuição concentrada em 5.0 e 10.0, sugerindo uma variável com poucas categorias efetivas.
- **IPS** (Indicador Psicossocial) apresenta distribuição relativamente concentrada em torno de 7.5.

---
## 5. Notas nas Disciplinas

In [ ]:
disciplinas = ['Matem', 'Portug', 'Inglês']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#264653', '#2a9d8f', '#e9c46a']

for i, col in enumerate(disciplinas):
    data = df[col].dropna()
    sns.histplot(data, bins=20, kde=True, ax=axes[i], color=colors[i])
    axes[i].set_title(f'{col} (n={len(data)})')
    axes[i].axvline(data.mean(), color='red', linestyle='--',
                    label=f'Média: {data.mean():.1f}')
    axes[i].axvline(data.median(), color='orange', linestyle='-.',
                    label=f'Mediana: {data.median():.1f}')
    axes[i].legend(fontsize=9)

plt.suptitle('Distribuição das Notas por Disciplina', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Comparação das disciplinas por gênero
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(disciplinas):
    sns.boxplot(x='Gênero', y=col, data=df, ax=axes[i], palette=['#e76f51', '#2a9d8f'])
    axes[i].set_title(f'{col} por Gênero')

plt.suptitle('Notas por Disciplina e Gênero', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

**Observações:**
- **Inglês** tem apenas 283 registros (33% da base), pois não é oferecida em todas as fases.
- As médias das disciplinas são similares (~5.8–6.3), com **Português** apresentando a maior mediana.
- **Matemática** apresenta distribuição mais uniforme com ligeira concentração em notas baixas.
- Não há diferença significativa aparente entre gêneros nas notas das disciplinas.

---
## 6. Análise da Variável-Alvo — Defasagem Escolar

In [ ]:
# 6.1 Distribuição da coluna Defas (valor numérico)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

defas_counts = df['Defas'].value_counts().sort_index()
colors_defas = ['#d62828' if x <= -3 else '#f77f00' if x < 0 else '#2a9d8f' for x in defas_counts.index]

defas_counts.plot(kind='bar', ax=axes[0], color=colors_defas)
axes[0].set_title('Distribuição de Defasagem (Defas)')
axes[0].set_xlabel('Valor de Defasagem')
axes[0].set_ylabel('Contagem')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=10)

# 6.2 Criar a variável-alvo categorizada
def classificar_defasagem(valor):
    """Classifica o nível de defasagem com base no valor numérico."""
    if valor >= 0:
        return 'baixo'
    elif valor >= -2:
        return 'medio'
    else:
        return 'alto'

df['nivel_defasagem'] = df['Defas'].apply(classificar_defasagem)

# Garantir a ordem
ordem = ['baixo', 'medio', 'alto']
cores_nivel = {'baixo': '#2a9d8f', 'medio': '#f4a261', 'alto': '#e76f51'}

nivel_counts = df['nivel_defasagem'].value_counts().reindex(ordem)
nivel_counts.plot(kind='pie', autopct='%1.1f%%', ax=axes[1],
                   colors=[cores_nivel[x] for x in ordem], startangle=90)
axes[1].set_title('Nível de Defasagem (target)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print('\nDistribuição do target:')
print(nivel_counts)
print(f'\nTotal: {nivel_counts.sum()}')

**Observações críticas sobre a variável-alvo:**
- A classe **"medio"** (defasagem de -1 a -2) é a mais frequente (~66% dos casos).
- A classe **"baixo"** (sem defasagem, Defas ≥ 0) representa ~30% dos alunos.
- A classe **"alto"** (defasagem ≤ -3) é a **minoritária** (~3%) — desbalanceamento significativo.
- Esse desbalanceamento justifica o uso de `class_weight='balanced'` no modelo de classificação.
- A concentração em Defas = -1 mostra que a maioria dos alunos está uma fase atrás da ideal.

In [ ]:
# 6.3 Defasagem por Gênero
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ct = pd.crosstab(df['Gênero'], df['nivel_defasagem'])[ordem]
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

ct.plot(kind='bar', ax=axes[0], color=[cores_nivel[x] for x in ordem])
axes[0].set_title('Defasagem por Gênero (contagem)')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Nível')

ct_pct.plot(kind='bar', stacked=True, ax=axes[1], color=[cores_nivel[x] for x in ordem])
axes[1].set_title('Defasagem por Gênero (% proporcional)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylabel('%')
axes[1].legend(title='Nível')

plt.tight_layout()
plt.show()

In [ ]:
# 6.4 Defasagem por Fase
fig, ax = plt.subplots(figsize=(12, 5))

ct_fase = pd.crosstab(df['Fase'], df['nivel_defasagem'])[ordem]
ct_fase_pct = ct_fase.div(ct_fase.sum(axis=1), axis=0) * 100

ct_fase_pct.plot(kind='bar', stacked=True, ax=ax, color=[cores_nivel[x] for x in ordem])
ax.set_title('Proporção de Defasagem por Fase')
ax.set_xlabel('Fase')
ax.set_ylabel('% dos alunos')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Nível de Defasagem', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 6.5 Defasagem por Instituição de Ensino
fig, ax = plt.subplots(figsize=(10, 5))

ct_inst = pd.crosstab(df['Instituição de ensino'], df['nivel_defasagem'])[ordem]
ct_inst_pct = ct_inst.div(ct_inst.sum(axis=1), axis=0) * 100

ct_inst_pct.plot(kind='bar', stacked=True, ax=ax, color=[cores_nivel[x] for x in ordem])
ax.set_title('Proporção de Defasagem por Instituição de Ensino')
ax.set_ylabel('% dos alunos')
ax.tick_params(axis='x', rotation=15)
ax.legend(title='Nível', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

In [ ]:
# 6.6 Indicadores por nível de defasagem
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(indicadores):
    sns.boxplot(x='nivel_defasagem', y=col, data=df, order=ordem,
                palette=cores_nivel, ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('')

axes[-1].set_visible(False)
plt.suptitle('Indicadores de Desempenho por Nível de Defasagem', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

**Observações:**
- O **INDE 22** diminui claramente com o aumento da defasagem — boa variável preditiva.
- **IDA** e **IEG** mostram diferenciação consistente entre os três níveis.
- Alunos com defasagem alta possuem medianas mais baixas em todos os indicadores.
- **IAA** (auto-avaliação) e **IAN** são menos discriminativos, mas ainda contribuem.
- As fases mais avançadas (5, 6, 7) concentram mais alunos com defasagem, o que é esperado.

---
## 7. Correlações e Relações Multivariadas

In [ ]:
# Selecionar apenas colunas numéricas relevantes
cols_numericas = ['Fase', 'Idade 22', 'Ano ingresso', 'INDE 22', 'Cg', 'Cf', 'Ct',
                  'Nº Av', 'IAA', 'IEG', 'IPS', 'IDA', 'Matem', 'Portug',
                  'IPV', 'IAN', 'Defas']

corr_matrix = df[cols_numericas].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1, annot_kws={'size': 9})
ax.set_title('Matriz de Correlação — Variáveis Numéricas', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlações com a variável-alvo (Defas)
corr_target = corr_matrix['Defas'].drop('Defas').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors_corr = ['#2a9d8f' if v > 0 else '#e76f51' for v in corr_target]
corr_target.plot(kind='barh', ax=ax, color=colors_corr)
ax.set_title('Correlação das Variáveis com Defasagem (Defas)')
ax.set_xlabel('Correlação de Pearson')
ax.axvline(0, color='black', linewidth=0.5)
for i, v in enumerate(corr_target):
    ax.text(v + 0.01 if v >= 0 else v - 0.06, i, f'{v:.2f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

**Principais correlações com a defasagem:**
- Correlação **positiva** (reduz defasagem): INDE 22, IDA, Matem, Portug, IEG, IPS — bons indicadores acadêmicos estão associados a menor defasagem.
- Correlação **negativa** (aumenta defasagem): Fase, Idade 22 — alunos mais velhos e em fases mais avançadas tendem a ter maior defasagem.
- **Cg** (contribuição da gestão) e **Cf** (contribuição familiar) apresentam correlações fracas.

In [ ]:
# Pairplot com as variáveis mais relevantes
cols_pairplot = ['INDE 22', 'IDA', 'IEG', 'IPS', 'Fase', 'nivel_defasagem']
g = sns.pairplot(df[cols_pairplot].dropna(), hue='nivel_defasagem',
                 hue_order=ordem, palette=cores_nivel,
                 diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20})
g.figure.suptitle('Pairplot — Indicadores mais Relevantes vs Defasagem', y=1.02, fontsize=14)
plt.show()

---
## 8. Evolução dos Alunos — Pedras (Classificação)

O sistema de **Pedras** classifica os alunos em quatro níveis de progressão:
- **Quartzo** (nível 1) → **Ágata** (nível 2) → **Ametista** (nível 3) → **Topázio** (nível 4)

In [ ]:
# Distribuição das Pedras por ano
pedra_cols = ['Pedra 20', 'Pedra 21', 'Pedra 22']
pedra_order = ['Quartzo', 'Ágata', 'Ametista', 'Topázio']
pedra_colors = {'Quartzo': '#adb5bd', 'Ágata': '#6c757d', 'Ametista': '#7b2d8e', 'Topázio': '#fca311'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(pedra_cols):
    data = df[col].dropna()
    counts = data.value_counts().reindex(pedra_order).fillna(0)
    counts.plot(kind='bar', ax=axes[i],
                color=[pedra_colors.get(x, '#ccc') for x in pedra_order])
    axes[i].set_title(f'{col} (n={len(data)})')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)
    for p in axes[i].patches:
        if p.get_height() > 0:
            axes[i].annotate(f'{int(p.get_height())}',
                            (p.get_x() + p.get_width()/2., p.get_height()),
                            ha='center', va='bottom', fontsize=10)

plt.suptitle('Distribuição das Pedras por Ano', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pedra 22 vs Nível de Defasagem
fig, ax = plt.subplots(figsize=(10, 5))

ct_pedra = pd.crosstab(df['Pedra 22'], df['nivel_defasagem'])[ordem]
ct_pedra = ct_pedra.reindex(pedra_order)
ct_pedra_pct = ct_pedra.div(ct_pedra.sum(axis=1), axis=0) * 100

ct_pedra_pct.plot(kind='bar', stacked=True, ax=ax,
                  color=[cores_nivel[x] for x in ordem])
ax.set_title('Nível de Defasagem por Pedra (2022)')
ax.set_ylabel('% dos alunos')
ax.set_xlabel('Pedra 2022')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Nível de Defasagem', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

**Observações sobre as Pedras:**
- **Quartzo** (nível mais baixo) concentra mais alunos com defasagem média e alta.
- **Topázio** (nível mais alto) é dominado por alunos sem defasagem (baixo).
- A progressão de Pedras reflete diretamente o desempenho e está fortemente associada à defasagem.
- Os anos 2020 e 2021 possuem muitos valores ausentes pois nem todos os alunos estavam no programa nesses anos.
- Em 2022, **Ametista** é a Pedra mais frequente, seguida de Ágata.

---
## 9. Comparação entre Abas (2022, 2023, 2024)

In [ ]:
# Comparar número de alunos e colunas entre anos
resumo_abas = []
for name, df_s in sheets.items():
    resumo_abas.append({
        'Aba': name,
        'Alunos': df_s.shape[0],
        'Colunas': df_s.shape[1],
        'Colunas exclusivas': len(set(df_s.columns) - set(sheets['PEDE2022'].columns)),
        'Missing total (%)': round(df_s.isnull().sum().sum() / (df_s.shape[0] * df_s.shape[1]) * 100, 1)
    })

pd.DataFrame(resumo_abas)

In [ ]:
# Colunas exclusivas de cada aba
cols_2022 = set(sheets['PEDE2022'].columns)
cols_2023 = set(sheets['PEDE2023'].columns)
cols_2024 = set(sheets['PEDE2024'].columns)

print('Colunas comuns a todas as abas:')
print(sorted(cols_2022 & cols_2023 & cols_2024))
print(f'\nColunas exclusivas de 2022: {sorted(cols_2022 - cols_2023 - cols_2024)}')
print(f'Colunas exclusivas de 2023: {sorted(cols_2023 - cols_2022 - cols_2024)}')
print(f'Colunas exclusivas de 2024: {sorted(cols_2024 - cols_2022 - cols_2023)}')

In [ ]:
# Distribuição de defasagem por ano
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
defas_col_map = {'PEDE2022': 'Defas', 'PEDE2023': 'Defasagem', 'PEDE2024': 'Defasagem'}

for i, (name, df_s) in enumerate(sheets.items()):
    col_defas = defas_col_map[name]
    vals = df_s[col_defas].value_counts().sort_index()
    colors = ['#d62828' if x <= -3 else '#f77f00' if x < 0 else '#2a9d8f' for x in vals.index]
    vals.plot(kind='bar', ax=axes[i], color=colors)
    axes[i].set_title(f'{name} (n={len(df_s)})')
    axes[i].set_xlabel('Defasagem')
    axes[i].tick_params(axis='x', rotation=0)

plt.suptitle('Distribuição de Defasagem por Ano', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Variáveis Complementares

In [ ]:
# Recomendação de Psicologia
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Rec Psicologia'].value_counts().plot(kind='bar', ax=axes[0],
                                          color=sns.color_palette('Set2'))
axes[0].set_title('Recomendação de Psicologia')
axes[0].tick_params(axis='x', rotation=30)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=10)

# Rec Psicologia vs Defasagem
ct_psi = pd.crosstab(df['Rec Psicologia'], df['nivel_defasagem'])[ordem]
ct_psi_pct = ct_psi.div(ct_psi.sum(axis=1), axis=0) * 100
ct_psi_pct.plot(kind='bar', stacked=True, ax=axes[1],
                color=[cores_nivel[x] for x in ordem])
axes[1].set_title('Defasagem por Recomendação Psicológica')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Nível', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()

In [ ]:
# Indicado e Atingiu PV (Ponto de Virada)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(['Indicado', 'Atingiu PV']):
    ct = pd.crosstab(df[col], df['nivel_defasagem'])[ordem]
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind='bar', stacked=True, ax=axes[i],
                color=[cores_nivel[x] for x in ordem])
    axes[i].set_title(f'Defasagem por {col}')
    axes[i].set_ylabel('%')
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].legend(title='Nível')

plt.tight_layout()
plt.show()

In [ ]:
# Componentes: Cg (contribuição da gestão), Cf (contribuição familiar), Ct (contribuição total)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(['Cg', 'Cf', 'Ct']):
    sns.boxplot(x='nivel_defasagem', y=col, data=df, order=ordem,
                palette=cores_nivel, ax=axes[i])
    axes[i].set_title(f'{col} por Nível de Defasagem')
    axes[i].set_xlabel('')

plt.suptitle('Componentes Cg, Cf, Ct por Nível de Defasagem', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
## 11. Resumo e Conclusões

### Principais achados da análise exploratória:

| Aspecto | Observação |
|---|---|
| **Base** | 860 alunos (2022), com 42 colunas incluindo identificadores, indicadores e notas |
| **Missing** | Inglês (~67%), Pedra 20/21 (~46-62%), Avaliador4 (~64%) — justificados pelo contexto |
| **Gênero** | Distribuição equilibrada (53% meninas, 47% meninos) |
| **Instituição** | 87% dos alunos em escolas públicas |
| **Target (Defas)** | Fortemente desbalanceado: ~30% baixo, ~67% médio, ~3% alto |
| **Indicadores** | INDE, IDA e IEG são os mais discriminativos para defasagem |
| **Pedras** | Classificação progressiva (Quartzo → Topázio) reflete diretamente o desempenho |
| **Correlações** | Fase e Idade correlacionam negativamente com Defas; indicadores acadêmicos correlacionam positivamente |

### Implicações para a modelagem:
1. **Desbalanceamento**: Necessário usar técnicas como `class_weight='balanced'` para dar mais peso à classe "alto"
2. **Leakage**: Colunas como INDE, Pedras encoded, Destaques e avaliadores devem ser removidas para evitar vazamento de informação do target
3. **Missing values**: Tratar com imputação — mediana para variáveis numéricas, moda para categóricas
4. **Feature engineering**: Criar variáveis derivadas como `tempo_no_programa`, `media_disciplinas`, `evolução de pedras`, `diff entre indicadores`
5. **Foco no recall da classe "alto"**: A classe minoritária representa os alunos com maior risco de defasagem severa e é a mais crítica para intervenção escolar